# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Big Data** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Examples on Structured Streaming (files)** </center>
---
**Profesor**: Pablo Camarillo Ramirez

# Create SparkSession

In [1]:
from pcamarillor.spark_utils import SparkUtils

su = SparkUtils("Structured Streaming with Files",
                   master_url="spark://spark-master:7077")

su.spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/09 03:44:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Create a data stream from a local socket

In [6]:
!mkdir -p /opt/spark/work-dir/data/streaming/logs/
!ls /opt/spark/work-dir/data/streaming/logs/

In [ ]:
import pyspark.sql.functions as F
from pathlib import Path
import shutil

logs_schema = SparkUtils.generate_schema([("raw_line", "string")])

input_path = "/opt/spark/work-dir/data/streaming/logs/"

# Create the stream
logs_df = (su.spark.readStream
            .format("text")
            .option("maxFilesPerTrigger", 1) # Let's process one file at a time
            .schema(logs_schema)
            .load(input_path))

# Transform original dataframe
parsed_df = (
    logs_df
    .withColumn("parts",     F.split(F.col("raw_line"), r" \| "))
    .withColumn("timestamp", F.to_timestamp(F.col("parts")[0], "yyyy-MM-dd HH:mm:ss"))
    .withColumn("level",     F.trim(F.col("parts")[1]))
    .withColumn("message",   F.trim(F.col("parts")[2]))
    .withColumn("server",    F.trim(F.col("parts")[3]))
    .drop("parts", "raw_line")               # keep only the clean columns
    .filter(F.col("timestamp").isNotNull())  # skip malformed lines
)

# Let's create a summary
summary_df = (
    parsed_df
    .groupBy("server", "level")
    .count()
    .orderBy("server", "level")
)

# Clean checkpoint
checkpoint_path = "/opt/spark/work-dir/checkpoints/logs_checkpoint"
dir_path = Path(checkpoint_path)
if dir_path.exists() and dir_path.is_dir():
    shutil.rmtree(dir_path)

# Write stream in the destination
query_events = (
    parsed_df.writeStream
    .outputMode("append")        # append: show new rows only
    .format("console")
    .option("truncate", False)   # don't cut off long messages
    .option("numRows", 20)
    .option("checkpointLocation", checkpoint_path)
    .queryName("parsed_logs")
    .start()
)

query_summary = (
    summary_df.writeStream
    .outputMode("complete")
    .format("console")
    .option("truncate", False)
    .queryName("summary_logs")
    .start()
)

print("   Press Ctrl+C to stop.\n")
su.spark.streams.awaitAnyTermination()


26/04/07 02:46:06 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/04/07 02:46:06 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-8d74f424-bc41-4726-b6e5-cb42a663c223. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/07 02:46:06 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


   Press Ctrl+C to stop.



-------------------------------------------
Batch: 0
-------------------------------------------
+-------------------+-----+---------------------------+-------------+
|timestamp          |level|message                    |server       |
+-------------------+-----+---------------------------+-------------+
|2026-04-07 02:53:56|INFO |Health check passed        |server-node-1|
|2026-04-07 02:54:03|WARN |CPU spike detected         |server-node-1|
|2026-04-07 02:54:12|ERROR|Disk full                  |server-node-4|
|2026-04-07 02:54:21|WARN |CPU spike detected         |server-node-1|
|2026-04-07 02:54:23|ERROR|500 Internal Server Error  |server-node-5|
|2026-04-07 02:54:32|INFO |Health check passed        |server-node-4|
|2026-04-07 02:54:37|INFO |Service restarted          |server-node-3|
|2026-04-07 02:54:44|ERROR|404 Not Found              |server-node-2|
|2026-04-07 02:54:52|ERROR|Database connection timeout|server-node-2|
|2026-04-07 02:54:59|INFO |Health check passed        |server-n

-------------------------------------------
Batch: 0
-------------------------------------------
+-------------+-----+-----+
|server       |level|count|
+-------------+-----+-----+
|server-node-1|INFO |3    |
|server-node-1|WARN |2    |
|server-node-2|ERROR|2    |
|server-node-3|INFO |1    |
|server-node-4|ERROR|1    |
|server-node-4|INFO |1    |
|server-node-5|ERROR|1    |
|server-node-5|INFO |1    |
+-------------+-----+-----+

-------------------------------------------
Batch: 1
-------------------------------------------
+-------------------+-----+-----------------------------+-------------+
|timestamp          |level|message                      |server       |
+-------------------+-----+-----------------------------+-------------+
|2026-04-07 02:54:16|ERROR|Database connection timeout  |server-node-2|
|2026-04-07 02:54:20|WARN |Disk usage 85%               |server-node-2|
|2026-04-07 02:54:28|INFO |Service restarted            |server-node-1|
|2026-04-07 02:54:33|WARN |Retry atte

### Connect Spark to the socket

In [5]:
su.spark.stop()